The code uses the code from youngsoul's pyimagesearch-covid19-image-classification: https://github.com/youngsoul/pyimagesearch-covid19-image-classification

it also uses the dataset from the COVID19, Pneumonia and Normal Chest X-ray PA Dataset: https://data.mendeley.com/datasets/mxc6vb7svm/2

this is to test the functionality of the code and the dataset.

In [ ]:
# cleans everything

%cd /content
#!rm -rf covid_pneumonia_dataset
!#rm -rf covid_pneumonia_dataset.zip
!#rm -rf covid_pneumonia_dataset_wrapped
!rm -rf pyimagesearch-covid19-image-classification

/content


In [ ]:
# Get the classifier code from GitHub
%cd /content
!git clone https://github.com/MarzSaod/pyimagesearch-covid19-image-classification
REPO = "/content/pyimagesearch-covid19-image-classification"
%cd {REPO}
!mkdir -p models model_performance
%cd /content

/content
Cloning into 'pyimagesearch-covid19-image-classification'...
remote: Enumerating objects: 481, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 481 (delta 16), reused 17 (delta 6), pack-reused 449 (from 1)
Receiving objects: 100% (481/481), 789.69 MiB | 34.51 MiB/s, done.
Resolving deltas: 100% (59/59), done.
Updating files: 100% (393/393), done.
/content/pyimagesearch-covid19-image-classification
/content


In [ ]:
# Downloads the dataset directly into the environment and unzips the dataset
%cd /content

mendeley_link = "https://data.mendeley.com/public-api/zip/mxc6vb7svm/download/2"

!wget -O covid_pneumonia_dataset.zip "$mendeley_link"

import zipfile
import os

with zipfile.ZipFile('covid_pneumonia_dataset.zip') as z:
    z.extractall('covid_pneumonia_dataset')

for item in os.listdir('covid_pneumonia_dataset'):
    if item.endswith('.zip'):
        inner_path = os.path.join('covid_pneumonia_dataset', item)
        print(f"Found the dataset inside: {item}")
        with zipfile.ZipFile(inner_path) as inner_z:
            print(f"contains {len(inner_z.namelist())} items")
            inner_z.extractall('covid_pneumonia_dataset')
        os.remove(inner_path)


/content
--2026-09-12 00:12:58--  https://data.mendeley.com/public-api/zip/mxc6vb7svm/download/2
Resolving data.mendeley.com (data.mendeley.com)... 162.159.133.86, 162.159.130.86
Connecting to data.mendeley.com (data.mendeley.com)|162.159.133.86|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/mxc6vb7svm-2.zip?X-Amz-Security-Token=IQoJb3JpZ2luX2VjEOD%2F%2F%2F%2F%2F%2F%2F%2F%2F%2FwEaCWV1LXdlc3QtMSJGMEQCIFX5Kzv1XrI%2ByPjqxgPH5a63x38T%2BmNjhSiLC2q5PdAHAiB4ZcTGYebUIDcyUTRxff%2BvjmHZ0jsiha8MlNN9rRZdjiqVBQio%2F%2F%2F%2F%2F%2F%2F%2F%2F%2F8BEAQaDDM2NzE0NzM4MzgyNSIMa6mn4vYmIKL658%2B6KukEINwd%2Bb6OyraFe%2F%2Fni6ED42I32jObcH1YBcY3irjWcupo8CjMNxx579kUKAHs94QWXcvcE1u8S3CZ1aqsMAKILwNIyl3brdWDRmn4qy49UIH6KcuB%2FCKFCcjVAv%2BacKCqyrNn00CHeWNu%2BWqrJF%2B8%2FnqbRC3SExGtMHVXUpLK7BXwRw0s9%2B2uBrmy1B2hCefWJ1xdFLzjJw92RdDEOywuEbIPJii53HeQwFe%2BEyEBrYNr1PG9e5TwNzxNDzPhnSouw7trTBjpmcIZ6LApWtxmM1CsAC1Ia0oGF31wDJrhO

In [ ]:
# to check the folders and image counts
import os

for root, dirs, files in os.walk("covid_pneumonia_dataset"):
    level = root.replace("covid_pneumonia_dataset", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

covid_pneumonia_dataset/  (0 files)
  pneumonia/  (2313 files)
  covid/  (2313 files)
  normal/  (2313 files)


In [ ]:
# to update folder configuration to match the github code

import shutil

wrapped_root = '/content/covid_pneumonia_dataset_wrapped/all'
os.makedirs(wrapped_root, exist_ok=True)

for cls in ['covid', 'normal', 'pneumonia']:
    src = f'/content/covid_pneumonia_dataset/{cls}'
    dst = f'{wrapped_root}/{cls}'
    if os.path.exists(src):
        shutil.move(src, dst)

print("Data is ready at:", wrapped_root)


Data is ready at: /content/covid_pneumonia_dataset_wrapped/all


In [ ]:
import tensorflow as tf
print("GPU devices:", tf.config.list_physical_devices('GPU'))

GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# test run with only 2 epochs just to confirm nothing errors out
%cd {REPO}
!python train_covid19.py --dataset /content/covid_pneumonia_dataset_wrapped/all --epochs 2

/content/pyimagesearch-covid19-image-classification
Running Train COVID19 Models
Using dataset directory: /content/covid_pneumonia_dataset_wrapped/all
[INFO] setting up data generators...
Found 5523 images belonging to 3 classes.
Found 1379 images belonging to 3 classes.
Class Labels: ['covid', 'normal', 'pneumonia']
2026-09-12 00:33:53.614009: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1789173233.615562   16417 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
---------------------------------------------------
Running Model: vgg16
[INFO] compiling model...
[INFO] training head...
Epoch 1/2
I0000 00:00:1789173253.687778   16463 device_compiler.h:196] Compiled cluster using XLA!  This line is logged a

In [ ]:
# run with 25 epochs
%cd {REPO}
!python train_covid19.py --dataset /content/covid_pneumonia_dataset_wrapped/all --epochs 25

/content/pyimagesearch-covid19-image-classification
Running Train COVID19 Models
Using dataset directory: /content/covid_pneumonia_dataset_wrapped/all
[INFO] setting up data generators...
Found 5523 images belonging to 3 classes.
Found 1379 images belonging to 3 classes.
Class Labels: ['covid', 'normal', 'pneumonia']
2026-09-12 00:49:40.491124: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1789174180.492649   20758 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
---------------------------------------------------
Running Model: vgg16
[INFO] compiling model...
[INFO] training head...
Epoch 1/25
I0000 00:00:1789174189.198617   20807 device_compiler.h:196] Compiled cluster using XLA!  This line is logged 